In [1]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.dataframe_functions import *
from src.yolk_functions import *
from src.sprayer import Sprayer
sprayer = Sprayer()

import statsmodels.formula.api as smf

import pyfixest as pf

from collections import defaultdict

## Load Data

In [2]:
with open('parliament_bib.pkl', 'rb') as file:
    parliament_bib = pickle.load(file)

print("Number of Parliaments: ", len(parliament_bib))

keys_to_delete = []

for key in parliament_bib.keys():
    eco = parliament_bib[key]["member"]["stance_eco"]
    soc = parliament_bib[key]["member"]["stance_soc"]
    elec_year = parliament_bib[key]["election_date"].year

    if pd.isna(parliament_bib[key]["cabinet"]["minority_cab"]):
        keys_to_delete.append(key)
        continue
    
    if len(eco) == 0:
        keys_to_delete.append(key)
    elif pd.isna(eco).any():
        keys_to_delete.append(key)
    elif len(soc) == 0:
        keys_to_delete.append(key)
    elif pd.isna(soc).any():
        keys_to_delete.append(key)

parliament_bib_selection = {k: v for k, v in parliament_bib.items() if k not in keys_to_delete}

minority_cabs = [p for i, p in parliament_bib_selection.items() if p["cabinet"]["minority_cab"]== 1]# and p["cabinet"]["minority_formal"] == 0]

print("Number of Parliaments with Valid Data: ", len(parliament_bib_selection))

print("Number of Minortiy Cabinets with Stance Data: ", len(minority_cabs))

Number of Parliaments:  618
Number of Parliaments with Valid Data:  584
Number of Minortiy Cabinets with Stance Data:  196


## Build Dataframe

In [3]:
agenda_control_dict = {'Austria': -0.044,
                       'Belgium': -0.17,
                       'Denmark': -0.106,
                       'Finland': -0.148,
                       'France': 0.333,
                       'Germany': -0.126,
                       'Greece': 0.28,
                       'Iceland': -0.17,
                       'Ireland': 0.519,
                       'Italy': -0.219,
                       'Luxembourg': -0.053,
                       'Netherlands': -0.527,
                       'Norway': -0.063,
                       'Portugal': 0.147,
                       'Spain': 0.221,
                       'Sweden': -0.427,
                       'United Kingdom': 0.69}

countries = []
dates = []
minority_cabinets = []
formal_minorities =[]
agenda_control_list = []
ppas = []
scps = []
scps_sq = []
ppa_names = []
enps = []
seat_share_largest_party_list = []
polarizations = []
bicam_list = []
pos_parl_list = []
is_denmark = []
is_norway = []
is_sweden = []
for key in parliament_bib_selection.keys():
    parliament = parliament_bib_selection[key]
    country = parliament["country"]
    countries.append(country)
    dates.append(parliament["election_date"])
    agenda_control_list.append(agenda_control_dict[country])
    minority_cabinets.append(parliament["cabinet"]["minority_cab"])
    formal_minorities.append(parliament["cabinet"]["minority_formal"])

    stances = parliament["member"]["stance"]
    yolk_center = parliament["yolk"]["center"]
    distances = [math.dist(s, yolk_center) for s in stances]
    ppa = min(distances)
    ppas.append(ppa)

    # ppa_index = np.argmin(distances)
    ppa_indices = np.where(np.array(distances) == np.array(distances).min())[0]
    if len(ppa_indices) == 1:
        ppa_index = ppa_indices[0]
    else:
        ppa_index = ppa_indices[np.argmax([parliament["member"]["seatshare"][p] for p in ppa_indices])]
    
    scp = parliament["member"]["seatshare"][ppa_index]
    scps.append(scp)
    scps_sq.append(scp**2)

    ppa_party = parliament["member"]["name"][ppa_index]
    ppa_names.append(ppa_party)

    enps.append(parliament["enp"])
    seat_share_largest_party_list.append(parliament["seat_share_largest_party"])
    polarizations.append(parliament["polarization"]["rile"])
    bicam_list.append(parliament["bicameralism"])
    pos_parl_list.append(parliament["positive_parliamentarism"])

    if country == "Denmark":
        is_denmark.append(1)
    else:
        is_denmark.append(0)
    if country == "Norway":
        is_norway.append(1)
    else:
        is_norway.append(0)
    if country == "Sweden":
        is_sweden.append(1)
    else:
        is_sweden.append(0)

df_dict = {"Country": countries,
           "Election_Date": dates,
           "Minority_Cabinet": minority_cabinets,
           "Formal_Minority": formal_minorities,
           "Agenda_Control": agenda_control_list,
           "PPA": ppas,
           "SCP": scps,
           "SCP_sq": scps_sq,
           "Centrist_Party": ppa_names,
           "ENP": enps,
           "Seat_Share_Largest_Party": seat_share_largest_party_list,
           "Seat_Share_Largest_Party_sq": np.array(seat_share_largest_party_list)**2,
           "Polarization": polarizations,
           "Bicameralism": bicam_list,
           "Positive_Parliamentarism": pos_parl_list,
           "Is_Denmark": is_denmark,
           "Is_Norway": is_norway,
           "Is_Sweden":is_sweden}

df = pd.DataFrame(df_dict)

In [4]:
df

,Country,Election_Date,Minority_Cabinet,Formal_Minority,Agenda_Control,PPA,SCP,SCP_sq,Centrist_Party,ENP,Seat_Share_Largest_Party,Seat_Share_Largest_Party_sq,Polarization,Bicameralism,Positive_Parliamentarism,Is_Denmark,Is_Norway,Is_Sweden
0,Austria,1949-10-09,0.0,0.0,-0.044,6.015172,0.096970,0.009403,VdU,2.544630,0.466667,0.217778,22.4869,1,0,0,0,0
1,Austria,1953-02-22,0.0,0.0,-0.044,12.416526,0.084848,0.007199,VdU,2.471181,0.448485,0.201139,22.2302,1,0,0,0,0
2,Austria,1956-05-13,0.0,0.0,-0.044,0.725036,0.036364,0.001322,FPÖ,2.223356,0.496970,0.246979,32.3879,1,0,0,0,0
3,Austria,1959-05-10,0.0,0.0,-0.044,11.357790,0.048485,0.002351,FPÖ,2.197514,0.478788,0.229238,20.1234,1,0,0,0,0
4,Austria,1959-05-10,0.0,0.0,-0.044,11.357790,0.048485,0.002351,FPÖ,2.197514,0.478788,0.229238,20.1234,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,United Kingdom,2017-06-08,1.0,0.0,0.690,2.192031,0.018462,0.000341,LibDems,2.475451,0.487692,0.237844,16.8186,1,0,0,0,0
580,United Kingdom,2019-12-12,0.0,0.0,0.690,1.010845,0.006154,0.000038,PC,2.392087,0.561538,0.315325,19.0413,1,0,0,0,0
581,United Kingdom,2019-12-12,0.0,0.0,0.690,1.010845,0.006154,0.000038,PC,2.392087,0.561538,0.315325,19.0413,1,0,0,0,0
582,United Kingdom,2019-12-12,0.0,0.0,0.690,1.010845,0.006154,0.000038,PC,2.392087,0.561538,0.315325,19.0413,1,0,0,0,0


## Model Calculations

In [6]:
formula1 = "Minority_Cabinet ~ Agenda_Control + Is_Denmark + Is_Norway + Is_Sweden"
formula2 = "Minority_Cabinet ~ Agenda_Control + PPA + SCP + SCP_sq + Agenda_Control * PPA + Agenda_Control * SCP + Agenda_Control * SCP_sq + PPA * SCP + PPA * SCP_sq + Agenda_Control * PPA * SCP + Agenda_Control * PPA * SCP_sq"
formula3 = "Minority_Cabinet ~ Agenda_Control + PPA + SCP + SCP_sq + Agenda_Control * PPA + Agenda_Control * SCP + Agenda_Control * SCP_sq + PPA * SCP + PPA * SCP_sq + Agenda_Control * PPA * SCP + Agenda_Control * PPA * SCP_sq + ENP + Seat_Share_Largest_Party + Seat_Share_Largest_Party_sq + Polarization + Bicameralism + Positive_Parliamentarism"
formula4 = "Minority_Cabinet ~ Agenda_Control + PPA + SCP + SCP_sq + Agenda_Control * PPA + Agenda_Control * SCP + Agenda_Control * SCP_sq + PPA * SCP + PPA * SCP_sq + Agenda_Control * PPA * SCP + Agenda_Control * PPA * SCP_sq + ENP + Seat_Share_Largest_Party + Seat_Share_Largest_Party_sq + Polarization + Bicameralism + Positive_Parliamentarism + Is_Denmark + Is_Norway + Is_Sweden"

model1 = smf.logit(formula1, data=df)
results1 = model1.fit()

model2 = smf.logit(formula2, data=df)
results2 = model2.fit()

model3 = smf.logit(formula3, data=df)
results3 = model3.fit()

model4 = smf.logit(formula4, data=df)
results4 = model4.fit()

print(results1.summary())

Optimization terminated successfully.
         Current function value: 0.529800
         Iterations 6
Optimization terminated successfully.
         Current function value: 0.614964
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.535162
         Iterations 7
Optimization terminated successfully.
         Current function value: 0.456465
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:       Minority_Cabinet   No. Observations:                  584
Model:                          Logit   Df Residuals:                      579
Method:                           MLE   Df Model:                            4
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                  0.1697
Time:                        10:51:25   Log-Likelihood:                -309.40
converged:                       True   LL-Null:                       -372.64
Covariance Type:            nonrobust  

## Export Model Results to Latex Code

In [7]:
def reg_results_to_latex(*results, r=3):
    for n, result in enumerate(results):
        names = result.params.index.tolist()
        params = result.params.tolist()
        stds = result.bse.tolist()
        pvalues = result.pvalues.tolist()

        vstrings = []
        names_df = []
        merge_column = []
        for name, param, std, p in zip(names, params, stds, pvalues):
            value_string = f"{round(param,r)}"
            if p < 0.001:
                value_string += r"\textsuperscript{***}"
            elif p < 0.01:
                value_string += r"\textsuperscript{**}"
            elif p < 0.05:
                value_string += r"\textsuperscript{*}"
            elif p < 0.1:
                value_string += r"\textsuperscript{$\dagger$}"
            vstrings.append(value_string)
            vstrings.append(f"({round(std,r)})")
            names_df.append(name.replace(":", r" $\times$ ").replace("_", " "))
            names_df.append("")

        df_dict = {"variables": names_df, f"{n+1}": vstrings}
        if n > 0:
            result_df = pd.concat([result_df, pd.DataFrame(df_dict)], axis=1)
        else:
            result_df = pd.DataFrame(df_dict)
    result_df = result_df.loc[:, ~result_df.columns.duplicated(keep="last")]
    variable_col = result_df.pop('variables')
    result_df.insert(0, 'variables', variable_col)
    result_df = result_df.fillna("")

    latex_string = result_df.to_latex(float_format="%.3f",index=False)
    print(latex_string)
    return result_df
dfres = reg_results_to_latex(results1, results2, results3, results4)

\begin{tabular}{lllll}
\toprule
variables & 1 & 2 & 3 & 4 \\
\midrule
Intercept & -0.707\textsuperscript{***} & 0.179 & -23.613\textsuperscript{***} & -11.176\textsuperscript{**} \\
 & (0.089) & (0.296) & (3.412) & (3.907) \\
Agenda Control & -0.635\textsuperscript{*} & -0.624 & 0.656 & 0.572 \\
 & (0.302) & (0.891) & (1.063) & (1.14) \\
PPA &  & -0.115\textsuperscript{$\dagger$} & -0.16\textsuperscript{*} & -0.245\textsuperscript{**} \\
 &  & (0.069) & (0.078) & (0.09) \\
SCP &  & -11.486\textsuperscript{**} & -18.038\textsuperscript{***} & -23.922\textsuperscript{***} \\
 &  & (4.058) & (4.789) & (5.268) \\
SCP sq &  & 20.358\textsuperscript{*} & 32.117\textsuperscript{**} & 43.528\textsuperscript{***} \\
 &  & (8.825) & (10.348) & (11.202) \\
Agenda Control $\times$ PPA &  & -0.106 & -0.208 & 0.044 \\
 &  & (0.235) & (0.265) & (0.275) \\
Agenda Control $\times$ SCP &  & 10.952 & -1.434 & 3.868 \\
 &  & (13.234) & (15.868) & (16.356) \\
Agenda Control $\times$ SCP sq &  & -35.939 & -